In [1]:
%pip install snorkel

   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.5 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.5 MB 824.7 kB/s eta 0:00:07
   --- ------------------------------------ 0.5/5.5 MB 824.7 kB/s eta 0:00:07
   ----- ---------------------------------- 0.8/5.5 MB 808.8 kB/s eta 0:00:06
   ------- -------------------------------- 1.0/5.5 MB 858.1 kB/s eta 0:00:06
   --------- ------------------------------ 1.3/5.5 MB 908.5 kB/s eta 0:00:05
   ----------- ---------------------------- 1.6/5.5 MB 925.1 kB/s eta 0:00:05
   ------------- -------------------------- 1.8/5.5 MB 975.9 kB/s eta 0:00:04
   --------------- ------------------------ 2.1/5.5 MB 1.0 MB/s eta 0:00:04
   ------------------- -------------------- 2.6/5.5 MB 1.1 MB/s eta 0:00:03
   -------------------- ------------------- 2.9/5.5 MB 1.1 MB/s eta 0:00:03
   ---------------------- ----------------- 3.1/5.5 MB 1.1 MB/s eta 0:00:03
   ---------

In [2]:
import pandas as pd

In [5]:
df = pd.read_csv(r'C:\Users\Damil\Downloads\Compressed\TMDB Top 10,000 Movies Dataset\TMDB-Movies-Dataset.csv', encoding = 'latin-1')

In [7]:
df.head()

,Unnamed: 0,id,title,overview,original_language,release_date,genre_ids,popularity,vote_average,vote_count,release_year,release_month
0,0,1560520,Batman: Knightfall Part 1: Knightfall,"Arkham Asylum has been destroyed, and all its ...",en,2026-06-23,"[16, 28, 12, 18]",26.0421,9.100,509,2026.0,6.0
1,1,980431,Avatar Aang: The Last Airbender,"Avatar Aang, the world's last Airbender, learn...",en,2026-07-24,"[16, 12, 14, 28]",48.0108,8.995,1442,2026.0,7.0
2,2,1007757,Swapped,"A small woodland creature and a majestic bird,...",en,2026-05-01,"[12, 16, 10751, 14]",27.3447,8.831,2301,2026.0,5.0
3,3,1632181,Accidental Partners,Two women discover they were both scammed by t...,es,2026-03-12,"[35, 10749]",10.8839,8.791,411,2026.0,3.0
4,4,1311031,Demon Slayer: Kimetsu no Yaiba Infinity Castle,The Demon Slayer Corps are drawn into the Infi...,ja,2025-07-18,"[16, 28, 14]",32.4054,8.800,2387,2025.0,7.0


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         10000 non-null  int64  
 1   id                 10000 non-null  int64  
 2   title              10000 non-null  str    
 3   overview           9998 non-null   str    
 4   original_language  10000 non-null  str    
 5   release_date       9997 non-null   str    
 6   genre_ids          10000 non-null  str    
 7   popularity         10000 non-null  float64
 8   vote_average       10000 non-null  float64
 9   vote_count         10000 non-null  int64  
 10  release_year       9997 non-null   float64
 11  release_month      9997 non-null   float64
dtypes: float64(4), int64(3), str(5)
memory usage: 3.8 MB


In [10]:
x = df.iloc[:,:-1]
y = df['popularity']

In [12]:
from snorkel.labeling import labeling_function

In [17]:


NEGATIVE = 0

@labeling_function()

def if_negative_keywords(row):
    return NEGATIVE if "disappointed" in row.text.lower() else -1

In [18]:
df.max()

Unnamed: 0                                                        9999
id                                                             1632181
title                                          âWuthering Heightsâ
overview             â2048: Nowhere to Runâ takes place one yea...
original_language                                                   zh
release_date                                                2026-10-19
genre_ids                                                           []
popularity                                                    550.7621
vote_average                                                       9.1
vote_count                                                       41879
release_year                                                    2026.0
release_month                                                     12.0
dtype: object

In [19]:
df.min()

Unnamed: 0                                                           0
id                                                                   2
title                                                           #Alive
overview             "Dirty" Harry Callahan is a San Francisco Poli...
original_language                                                   ar
release_date                                                1902-06-15
genre_ids                                       [10402, 10749, 18, 35]
popularity                                                      0.0044
vote_average                                                       5.7
vote_count                                                         300
release_year                                                    1902.0
release_month                                                      1.0
dtype: object

In [31]:
##POPULARITY RULE

ABSTAIN = -1
not_popular = 0
popular = 1

@labeling_function()

def popularity(record):
    if record['popularity'] < 550 and record['popularity'] > 30:
       return popular
    elif record['popularity'] < 30:
        return not_popular
    else:
        return ABSTAIN

In [54]:
##Vote_averge_rule

@labeling_function()
def vote_average(record):
    if record['vote_average'] >= 7.5:
        return popular
    elif record['vote_average'] < 5:
        return not_popular
    else:
        return ABSTAIN


In [21]:
##YEAR RULE

@labeling_function()

def release_year(record):
    if record['release_year'] > 2018:
       return popular
    else:
       return not popular

In [55]:
##Creating a labelling function
z = [popularity, release_year, vote_average]

from snorkel.labeling import PandasLFApplier
applier = PandasLFApplier(lfs = z)
train = applier.apply(df=x)

100%|██████████| 10000/10000 [00:01<00:00, 5569.80it/s]


In [56]:
print(train.shape)

(10000, 3)


In [57]:
print(train[:5])

[[0 1 1]
 [1 1 1]
 [0 1 1]
 [0 1 1]
 [1 1 1]]


In [43]:
from snorkel.labeling.model import LabelModel
label_mode = LabelModel(cardinality=2, verbose=True)

In [58]:
##Train the label
label_mode.fit(L_train=train, n_epochs = 1000 , seed = 100)

INFO:root:Computing O...
INFO:root:Estimating \mu...
  2%|▎         | 25/1000 [00:02<00:55, 17.45epoch/s]INFO:root:[30 epochs]: TRAIN:[loss=0.058]
INFO:root:[40 epochs]: TRAIN:[loss=0.012]
INFO:root:[50 epochs]: TRAIN:[loss=0.014]
  5%|▌         | 54/1000 [00:02<00:20, 45.65epoch/s]INFO:root:[60 epochs]: TRAIN:[loss=0.006]
INFO:root:[70 epochs]: TRAIN:[loss=0.004]
INFO:root:[80 epochs]: TRAIN:[loss=0.003]
  8%|▊         | 82/1000 [00:02<00:12, 75.65epoch/s]INFO:root:[90 epochs]: TRAIN:[loss=0.003]
INFO:root:[100 epochs]: TRAIN:[loss=0.002]
INFO:root:[110 epochs]: TRAIN:[loss=0.002]
 11%|█▏        | 113/1000 [00:02<00:07, 111.87epoch/s]INFO:root:[120 epochs]: TRAIN:[loss=0.002]
INFO:root:[130 epochs]: TRAIN:[loss=0.001]
 14%|█▎        | 137/1000 [00:02<00:06, 128.52epoch/s]INFO:root:[140 epochs]: TRAIN:[loss=0.001]
INFO:root:[150 epochs]: TRAIN:[loss=0.001]
INFO:root:[160 epochs]: TRAIN:[loss=0.001]
 16%|█▌        | 162/1000 [00:02<00:05, 151.98epoch/s]INFO:root:[170 epochs]: TRAIN:[los

In [61]:
x['Labels'] = label_mode.predict(L=train)
x['Labels']

0       1
1       1
2       1
3       1
4       1
       ..
9995    0
9996    0
9997    0
9998    0
9999    0
Name: Labels, Length: 10000, dtype: int64